In [1]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_infix
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = False
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix("^GSPC", index_dict, all_stocks) 

# Modify the current date
current_date = modify_current_date(start, index_name)

In [2]:
# Index
indices = ["^HSI", "^GSPC", "^IXIC", "QLD", "TQQQ"]
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite", "QLD": "ProShares Ultra QQQ", "TQQQ": "ProShares UltraPro QQQ"}

# Momentum ETFs
etfs_dict = {
    "MTUM": "iShares MSCI USA Momentum Factor ETF",
    "IMTM": "iShares MSCI Intl Momentum Factor ETF",
    "SMLF": "iShares U.S. Small‑Cap Equity Factor ETF",
    "IUSV": "iShares S&P U.S. Value ETF",
    "IJJ": "iShares S&P Mid-Cap 400 Value ETF",
    "IJS": "iShares S&P Small-Cap 600 Value ETF",
    "EFV": "iShares MSCI EAFE Value ETF",
    "SPMO": "Invesco S&P 500 Momentum ETF",
    "XMMO": "Invesco S&P MidCap Momentum ETF",
    "XSMO": "Invesco S&P SmallCap Momentum ETF",
    "PDP": "Invesco Dorsey Wright Momentum ETF",
    "IDMO": "Invesco S&P Intl Developed Momentum ETF",
    "RPV": "Invesco S&P 500 Pure Value ETF",
    "SPHD": "Invesco S&P 500 High Dividend Low Volatility ETF",
    "VFMO": "Vanguard U.S. Momentum Factor ETF",
    "VYM": "Vanguard High Dividend Yield ETF",
    "VOE": "Vanguard S&P Mid-Cap Value ETF",
    "VONV": "Vanguard Russell 1000 Value ETF",
    "VOOV": "Vanguard S&P 500 Value ETF",
    "MGV": "Vanguard Mega Cap Value ETF",
    "VBR": "Vanguard Small-Cap Value ETF",
    "VTV": "Vanguard Value ETF",
    "AVLV": "Avantis U.S. Large Cap Value ETF",
    "AVUV": "Avantis U.S. Small Cap Value ETF",
    "JMOM": "JPMorgan U.S. Momentum Factor ETF",
    "GLOV": "Activebeta World Low Vol Plus Equity ETF",
    "DWMF": "WisdomTree International Multifactor ETF",
    "AFLG": "First Trust Active Factor Large Cap Intl ETF",
    "CGDV": "Capital Group Dividend Value ETF",
    "SCHD": "Schwab U.S. Dividend Equity ETF",
    "3070.HK": "Ping An of China CSI HK Dividend ETF"
}
etfs = list(etfs_dict.keys())

In [3]:
# Create an empty DataFrame to store the data of the ETFs
etfs_data = pd.DataFrame(columns=[
    "Symbol", "Establishment Date", "1 Year CAGR (%)", "3 Year CAGR (%)", 
    "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)", "Max Period CAGR (%)", 
    "Max Period Annual Volatility (%)", "Max Period Sharpe", 
    "Max Period Sortino", "Max Period MDD (%)", "Max Period Calmar", 
    "5 Year Annual Volatility (%)", "5 Year Sharpe", "5 Year Sortino", 
    "5 Year MDD (%)", "5 Year Calmar", "5 Year Peak Date", 
    "5 Year Recovery Date", "5 Year Recovery Period (days)"
])

# Loop through each ETF in the indices and momentum ETFs
for etf in tqdm(indices + etfs):
    # Get the name of the ETF
    etf_name = {**index_dict, **etfs_dict}.get(etf, etf)
    etfs_data.loc[etf_name, "Symbol"] = etf

    # Get the dataframe for the ETF
    df = get_df(etf, current_date, max_period=True, adj=True)
    
    # Get the establishment date of the ETF
    establishment_date = df.index[0].date()
    etfs_data.loc[etf_name, "Establishment Date"] = establishment_date
    
    # Calculate CAGR values for different periods
    for period, label in zip([252, 252 * 3, 252 * 5, 252 * 10, 252 * 20], 
                             ["1 Year CAGR (%)", "3 Year CAGR (%)", "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)"]):
        if len(df) >= period:
            start_price = df["Close"].iloc[- period]
            end_price = df["Close"].iloc[- 1]
            cagr = ((end_price / start_price) ** (1 / (period / 252)) - 1) * 100
            etfs_data.loc[etf_name, label] = cagr

    # Calculate CAGR of maximum period
    start_price = df["Close"].iloc[0]
    end_price = df["Close"].iloc[- 1]
    max_period_cagr = ((end_price / start_price) ** (1 / (len(df) / 252)) - 1) * 100
    etfs_data.loc[etf_name, "Max Period CAGR (%)"] = max_period_cagr

    # Calculate percent change and cumulative return
    df["Percent Change"] = df["Close"].pct_change().dropna()
    df["Cumulative Return"] = (1 + df["Percent Change"]).cumprod()
    daily_returns = df["Percent Change"]
    cumulative_returns = df["Cumulative Return"]
        
    # Calculate annual volatility
    annual_volatility = daily_returns.std() * np.sqrt(252) * 100
    etfs_data.loc[etf_name, "Max Period Annual Volatility (%)"] = annual_volatility

    # Calculate 5 year annual volatility
    if len(df) >= 252 * 5:
        daily_returns_5y = daily_returns.iloc[- 252 * 5:]
        annual_volatility_5y = daily_returns_5y.std() * np.sqrt(252) * 100
        etfs_data.loc[etf_name, "5 Year Annual Volatility (%)"] = annual_volatility_5y

    # Calculate Sharpe ratio
    risk_free_rate = 0.03  # Assuming a risk-free rate of 3%
    sharpe_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (daily_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sharpe"] = sharpe_ratio

    # Calculate 5 year Sharpe ratio
    if len(df) >= 252 * 5:
        sharpe_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (daily_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sharpe"] = sharpe_ratio_5y

    # Calculate Sortino ratio
    negative_returns = daily_returns[daily_returns < 0]
    sortino_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (negative_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sortino"] = sortino_ratio

    # Calculate 5 year Sortino ratio
    if len(df) >= 252 * 5:
        negative_returns_5y = daily_returns_5y[daily_returns_5y < 0]
        sortino_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (negative_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sortino"] = sortino_ratio_5y

    # Calculate maximum drawdown
    df["Peak"] = df["Cumulative Return"].cummax()
    df["Drawdown"] = (df["Cumulative Return"] - df["Peak"]) / df["Peak"]
    drawdowns = df["Drawdown"]
    mdd = drawdowns.min() * 100
    mdd_date = drawdowns.idxmin()
    etfs_data.loc[etf_name, "Max Period MDD (%)"] = mdd

    # Calculate 5 year maximum drawdown
    if len(df) >= 252 * 5:
        drawdowns_5y = drawdowns[- 252 * 5:]
        mdd_5y = drawdowns_5y.min() * 100
        etfs_data.loc[etf_name, "5 Year MDD (%)"] = mdd_5y
        mdd_date_5y = drawdowns_5y.idxmin()

        # Find the peak date before the maximum drawdown
        peak_mask_5y = (df.index <= mdd_date_5y) & (df["Cumulative Return"] == df["Peak"])
        peak_date_5y = df.index[peak_mask_5y][-1]
        etfs_data.loc[etf_name, "5 Year Peak Date"] = peak_date_5y.date()

        # Find the recovery date after the maximum drawdown
        rec_mask_5y = (df.index > mdd_date_5y) & (df["Cumulative Return"] >= df.loc[peak_date_5y, "Cumulative Return"])
        rec_date_5y = df.index[rec_mask_5y][0] if len(df.index[rec_mask_5y]) > 0 else None
        if rec_date_5y is not None:
            etfs_data.loc[etf_name, "5 Year Recovery Date"] = rec_date_5y.date()
            rec_period_5y = (rec_date_5y - peak_date_5y).days
            etfs_data.loc[etf_name, "5 Year Recovery Period (days)"] = rec_period_5y

        # Calculate the two most significant maximum drawdown in the 5 year period
        drawdowns_5y = drawdowns_5y.sort_values()
        mdd_5y = drawdowns_5y.iloc[0] * 100

    # Calculate Calmar ratio
    calmar_ratio = (daily_returns.mean() * 252 - risk_free_rate) / abs(mdd / 100)
    etfs_data.loc[etf_name, "Max Period Calmar"] = calmar_ratio

    # Calculate 5 year Calmar ratio
    if len(df) >= 252 * 5:
        calmar_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / abs(mdd_5y / 100)
        etfs_data.loc[etf_name, "5 Year Calmar"] = calmar_ratio_5y

100%|██████████| 36/36 [00:18<00:00,  1.93it/s]


In [4]:
etfs_data

,Symbol,Establishment Date,1 Year CAGR (%),3 Year CAGR (%),5 Year CAGR (%),10 Year CAGR (%),20 Year CAGR (%),Max Period CAGR (%),Max Period Annual Volatility (%),Max Period Sharpe,...,Max Period MDD (%),Max Period Calmar,5 Year Annual Volatility (%),5 Year Sharpe,5 Year Sortino,5 Year MDD (%),5 Year Calmar,5 Year Peak Date,5 Year Recovery Date,5 Year Recovery Period (days)
Hang Seng Index,^HSI,1986-12-31,34.189623,4.321881,0.899532,-1.359077,3.002914,6.123828,25.695225,0.245194,...,-65.18186,0.096657,25.180595,0.036772,0.054873,-55.700772,0.016624,2018-01-26,NaN,NaN
S&P 500,^GSPC,1927-12-30,11.108446,17.057767,14.7568,11.708106,8.610422,6.225142,18.970543,0.255289,...,-86.189579,0.05619,17.394245,0.693404,0.955551,-25.425097,0.474383,2022-01-03,2024-01-19,746
NASDAQ Composite,^IXIC,1971-02-05,10.393272,20.995297,14.756061,15.220371,12.173749,10.27694,20.178559,0.437281,...,-77.932386,0.113223,23.21126,0.572272,0.80741,-36.39528,0.364969,2021-11-19,2024-02-29,832
ProShares Ultra QQQ,QLD,2006-06-21,8.155554,37.791157,24.716347,29.252062,NaN,24.035627,44.301687,0.641264,...,-83.12887,0.341747,46.821757,0.635608,0.899101,-63.684482,0.467308,2021-11-19,2024-05-24,917
ProShares UltraPro QQQ,TQQQ,2010-02-11,-0.207345,46.502579,26.286565,34.538921,NaN,41.217089,61.537739,0.82358,...,-81.659848,0.620638,69.615428,0.634886,0.89668,-81.659848,0.541244,2021-11-19,2024-12-04,1111
iShares MSCI USA Momentum Factor ETF,MTUM,2013-04-18,18.402805,21.407574,12.883746,14.009906,NaN,14.777517,19.671155,0.647168,...,-34.082278,0.373524,21.319495,0.531205,0.734954,-32.284534,0.350788,2021-11-03,2024-03-04,852
iShares MSCI Intl Momentum Factor ETF,IMTM,2015-01-27,14.541608,17.799735,10.182877,7.818616,NaN,7.905561,20.277233,0.327781,...,-30.683156,0.216617,17.381694,0.465463,0.692874,-30.683156,0.26368,2021-11-08,2024-02-22,836
iShares U.S. Small‑Cap Equity Factor ETF,SMLF,2015-04-30,15.14653,14.763381,16.013109,9.909408,NaN,10.173699,21.889539,0.415889,...,-41.892211,0.217311,21.600666,0.637696,0.998134,-26.277099,0.524208,2024-11-25,NaN,NaN
iShares S&P U.S. Value ETF,IUSV,2000-08-04,10.643673,14.651334,15.393734,10.416875,8.577371,8.136084,19.248906,0.347109,...,-60.183504,0.111018,15.405638,0.794022,1.168761,-18.154398,0.673799,2020-01-17,2020-12-31,349
iShares S&P Mid-Cap 400 Value ETF,IJJ,2000-07-28,14.717961,12.028446,16.471527,9.125153,8.942346,10.133061,21.979717,0.413096,...,-58.003733,0.156537,20.720891,0.672619,1.062055,-25.05405,0.556288,2020-01-16,2020-12-04,323


In [5]:
# Save the data as a CSV file
result_folder = "Result"
filename = os.path.join(result_folder, "etfs_comparison.csv")
etfs_data.to_csv(filename)